# Strategy A architecture
# ───────────────────────
#  Encoder:
#    genes [5000] → MaskedLinear → pathway scores [n_pw]
#                → hidden [256] → latent z [32]
#
#  Metadata fusion (Strategy A — latent concatenation):
#    z [32] || one_hot(vaccination) [n_vax] || one_hot(variant) [n_var]
#    → z_meta [32 + n_vax + n_var]
#
#  Severity head:
#    z_meta → hidden [64] → severity score / class
#
#  Decoder (reconstruction — keeps AE honest):
#    z [32] → hidden [256] → pathway scores → genes [5000]
#
# Why concatenate to z and NOT to the input genes?

In [25]:
# %% ── Cell 1: CONFIG ─────────────────────────────────────────────────────────
 
CONFIG = {
    # ── Input ──────────────────────────────────────────────────────────────
    "h5ad_path":    r"C:\Users\ulago\Downloads\Variant_Vax_obj.h5ad",
 
    # ── Pilot mode ─────────────────────────────────────────────────────────
    "PILOT_MODE":        True,
    "pilot_fraction":    0.20,
    "pilot_seed":        42,
 
    # ── Column names (None = auto-detect) ──────────────────────────────────
    "groupby":           None,   # cell type / cluster
    "severity_col":      None,   # WHO score / severity — the prediction target
    "severity_type":     "continuous",  # "continuous" or "categorical"
    "patient_col":       "Participant",   # donor ID for leakage-free patient split
 
    # ── Metadata columns for Strategy A ────────────────────────────────────
    # Set to None to auto-detect; set to a string to force a specific column.
    # Set to False to explicitly disable that metadata channel.
    "vaccination_col":   None,   # e.g. "Vaccination_Status", "vaccinated"
    "variant_col":       None,   # e.g. "Variant", "covid_variant"
 
    # ── HVG ────────────────────────────────────────────────────────────────
    "use_hvg":           True,
    "n_hvg":             5000,
 
    # ── GO mask ────────────────────────────────────────────────────────────
    "gmt_libraries": [
        "GO_Biological_Process_2023",
        "GO_Molecular_Function_2023",
        "GO_Cellular_Component_2023",
    ],
    "gmt_cache_dir":          "gmt_cache",
    "min_genes_per_pathway":  5,
    "max_genes_per_pathway":  500,
    "top_n_pathways":         3000,
 
    # ── Architecture ───────────────────────────────────────────────────────
    "latent_dim":        32,
    "pathway_hidden":    256,
    "severity_hidden":   64,    # hidden units in severity head after concat
    "dropout":           0.3,
    "use_layernorm":     True,
 
    # ── Training ───────────────────────────────────────────────────────────
    "epochs":            100,
    "batch_size":        256,
    "lr":                1e-3,
    "weight_decay":      1e-4,
    "recon_weight":      2.0,   # weight on reconstruction loss
    "severity_weight":   1.0,   # weight on severity loss — higher = more focus
    "val_fraction":      0.15,
    "random_seed":       42,
    "early_stop_patience":   10,
    "early_stop_min_delta":  1e-4,
 
    # ── UMAP ───────────────────────────────────────────────────────────────
    "umap_n_neighbors":  15,
    "umap_min_dist":     0.3,
    "umap_n_sample":     500,   # cells sampled for Pearson/Spearman distance
 
    # ── SHAP ───────────────────────────────────────────────────────────────
    "shap_n_cells":      300,
 
    # ── Immune cell type filtering ────────────────────────────────────────
    # Restricts training to immune-relevant cell types only.
    # Non-immune cells (epithelial, RBC, stromal) carry no severity signal.
    # Set cell_type_col to None to auto-detect from obs columns.
    "immune_filter_enabled": True,
    "cell_type_col":         None,
    "immune_keywords": [
        "t cell", "t_cell", "tcell", "cd4", "cd8",
        "nk", "natural killer",
        "b cell", "b_cell", "bcell", "plasma",
        "monocyte", "macrophage", "myeloid",
        "dendritic", "dc ", " dc",
        "neutrophil", "basophil", "eosinophil", "mast",
        "innate", "lymphocyte", "pbmc",
    ],

    # ── Output ─────────────────────────────────────────────────────────────
    "outdir":            "strategy_a_results",
}
 
PILOT_OVERRIDES = {"epochs": 50, "batch_size": 128}
FULL_OVERRIDES  = {}

In [26]:
# %% ── Cell 2: Imports ────────────────────────────────────────────────────────
 
import sys, os, warnings, time, pickle
from pathlib import Path
warnings.filterwarnings("ignore")
 
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import scipy.sparse as sp
import scipy.stats as stats
from tqdm import tqdm
 
import anndata as ad
import scanpy as sc
import gseapy as gp
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset
 
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (r2_score, mean_squared_error,
                             accuracy_score, classification_report,
                             roc_auc_score)
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import KNeighborsClassifier
 
try:
    import shap; HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("SHAP not installed — pip install shap")
 
# ── Apply pilot / full overrides ──────────────────────────────────────────────
if CONFIG["PILOT_MODE"]:
    CONFIG.update(PILOT_OVERRIDES)
    RUN_TAG = "pilot"
    print(f"PILOT MODE — {int(CONFIG['pilot_fraction']*100)}% subsample, "
          f"{CONFIG['epochs']} epochs")
else:
    CONFIG.update(FULL_OVERRIDES)
    RUN_TAG = "full"
    print(f"FULL MODE  — {CONFIG['epochs']} epochs")
 
OUTDIR = Path(CONFIG["outdir"]) / RUN_TAG
OUTDIR.mkdir(parents=True, exist_ok=True)
 
def _device():
    if torch.cuda.is_available():         return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")
 
DEVICE = _device()
torch.manual_seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])
print(f"Device : {DEVICE}")
print(f"Output : {OUTDIR.resolve()}")

PILOT MODE — 20% subsample, 50 epochs
Device : cpu
Output : C:\Users\ulago\Downloads\Applied-Machine-Learning-Final-Project\strategy_a_results\pilot


In [27]:
# %% ── Cell 3: Load h5ad ──────────────────────────────────────────────────────
 
def auto_col(obs, keywords, exclude=None):
    """Return first obs column whose name contains any keyword (case-insensitive)."""
    for kw in keywords:
        for c in obs.columns:
            if kw.lower() in c.lower():
                if exclude and any(e.lower() in c.lower() for e in exclude):
                    continue
                return c
    return None
 
print("\n[1/10] Loading data …")
adata = sc.read_h5ad(CONFIG["h5ad_path"])
print(f"       {adata.n_obs:,} cells × {adata.n_vars:,} genes")
print(f"       obs columns: {list(adata.obs.columns)}")
 
if adata.raw is not None:
    adata = adata.raw.to_adata()
 
if adata.X.max() > 50:
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    print("       Normalised + log1p")
 
# ── Auto-detect columns ───────────────────────────────────────────────────────
obs = adata.obs
 
GROUPBY      = CONFIG["groupby"]      or auto_col(
    obs, ["leiden","louvain","cell_type","celltype","cluster","annotation"])
SEVERITY_COL = CONFIG["severity_col"] or auto_col(
    obs, ["severity","who","outcome","status","grade","score","disease"])
PATIENT_COL  = CONFIG["patient_col"]  or auto_col(
    obs, ["donor","patient","sample","subject","individual","participant"])
 
# Vaccination column
if CONFIG["vaccination_col"] is False:
    VAX_COL = None
else:
    VAX_COL = CONFIG["vaccination_col"] or auto_col(
        obs, ["vaccin","vax","immunis","boost"])
 
# Variant column
if CONFIG["variant_col"] is False:
    VAR_COL = None
else:
    VAR_COL = CONFIG["variant_col"] or auto_col(
        obs, ["variant","strain","omicron","delta","ancestral",
              "alpha","lineage","clade"])
 
print(f"\n       Groupby      : '{GROUPBY}'")
print(f"       Severity     : '{SEVERITY_COL}'")
print(f"       Patient      : '{PATIENT_COL}'")
print(f"       Vaccination  : '{VAX_COL}'")
print(f"       Variant      : '{VAR_COL}'")
 
if SEVERITY_COL is None:
    raise ValueError(
        "No severity column detected. Set CONFIG['severity_col'] explicitly.\n"
        f"Available obs columns: {list(obs.columns)}")
 
# ── Subsample ─────────────────────────────────────────────────────────────────
if CONFIG["PILOT_MODE"]:
    rng = np.random.default_rng(CONFIG["pilot_seed"])
    strat = GROUPBY
    if strat and strat in obs.columns:
        labels = obs[strat].values
        idx = np.concatenate([
            rng.choice(np.where(labels==s)[0],
                       size=max(1, round((labels==s).sum()*CONFIG["pilot_fraction"])),
                       replace=False)
            for s in np.unique(labels)])
        rng.shuffle(idx)
    else:
        idx = rng.choice(adata.n_obs,
                         size=int(adata.n_obs*CONFIG["pilot_fraction"]),
                         replace=False)
    adata = adata[idx].copy()
    print(f"\n       Pilot subsample: {adata.n_obs:,} cells")
 

# %% ── Immune cell type filtering ─────────────────────────────────────────────
# Retains only immune-relevant cell types. This removes cells (e.g. epithelial,
# RBC, stromal) whose GO pathway profiles carry no severity signal, reducing
# noise in the regression target.
#
# The filter is intentionally permissive — any cell type label containing
# one of the immune_keywords is kept. Unknown cell type columns fall back
# gracefully (all cells retained with a warning).

if CONFIG["immune_filter_enabled"]:
    # Resolve which column holds cell type labels
    ct_col = CONFIG["cell_type_col"] or GROUPBY or auto_col(
        adata.obs,
        ["cell_type", "celltype", "leiden", "louvain", "cluster",
         "annotation", "lineage", "cell.type"])

    if ct_col is None or ct_col not in adata.obs.columns:
        print(f"\n       [immune filter] WARNING: no cell type column found — "
              f"skipping filter (all {adata.n_obs:,} cells retained)")
        IMMUNE_COL_USED = None
    else:
        IMMUNE_COL_USED = ct_col
        ct_labels = adata.obs[ct_col].astype(str).str.lower().values
        keywords  = [kw.lower() for kw in CONFIG["immune_keywords"]]

        # A cell is kept if its label contains at least one keyword
        keep_mask = np.zeros(len(ct_labels), dtype=bool)
        for kw in keywords:
            keep_mask |= np.array([kw in lbl for lbl in ct_labels])

        n_before = adata.n_obs
        adata = adata[keep_mask].copy()
        n_after = adata.n_obs
        removed = n_before - n_after

        print(f"\n       [immune filter] Column : '{ct_col}'")
        print(f"       [immune filter] Before  : {n_before:,} cells")
        print(f"       [immune filter] Removed : {removed:,} non-immune cells "
              f"({removed/n_before*100:.1f}%)")
        print(f"       [immune filter] Kept    : {n_after:,} immune cells")

        # Show breakdown of retained cell types
        retained_types = adata.obs[ct_col].value_counts()
        print(f"       [immune filter] Retained cell types:")
        for ct, cnt in retained_types.items():
            print(f"         {ct:<35} {cnt:>6,} cells")

        # Check patients are still represented
        if PATIENT_COL and PATIENT_COL in adata.obs.columns:
            n_pts = adata.obs[PATIENT_COL].nunique()
            print(f"       [immune filter] Patients retained: {n_pts}")
else:
    IMMUNE_COL_USED = None
    print("\n       [immune filter] Disabled — all cell types used")

# ── HVG filter ────────────────────────────────────────────────────────────────
adata_full_genes = adata.copy()  # keep for reference UMAP
if CONFIG["use_hvg"] and adata.n_vars > CONFIG["n_hvg"]:
    sc.pp.highly_variable_genes(
        adata, n_top_genes=CONFIG["n_hvg"], flavor="seurat_v3")
    adata = adata[:, adata.var["highly_variable"]].copy()
    print(f"       HVG filter: {adata.n_vars:,} genes retained")


[1/10] Loading data …
       48,730 cells × 29,961 genes
       obs columns: ['nCount_RNA', 'nFeature_RNA', 'percent.mt', 'Variant_Group', 'Participant', 'SARSCoV2_PCR_Status', 'Vaccination_Status', 'WHO_Score_at_Peak', 'SingleCell_SARSCoV2_RNA_Status', 'Coarse_Annotation', 'Detailed_Annotation', 'Variant_Vax_Group']
       Normalised + log1p

       Groupby      : 'Coarse_Annotation'
       Severity     : 'WHO_Score_at_Peak'
       Patient      : 'Participant'
       Vaccination  : 'Vaccination_Status'
       Variant      : 'Variant_Group'

       Pilot subsample: 9,747 cells
       HVG filter: 5,000 genes retained


In [28]:
# %% ── Cell 4: Encode metadata ────────────────────────────────────────────────
# One-hot encode vaccination status and variant.
# These vectors will be concatenated to the latent z AFTER the encoder
# (Strategy A) — they never touch the GO-masked layer.
 
print("\n[2/10] Encoding metadata …")
 
meta_tensors = {}   # {name: torch.FloatTensor [n_cells, n_categories]}
meta_encoders = {}  # {name: LabelEncoder} — saved for interpretation
 
def encode_metadata_col(obs, col_name, label):
    if col_name is None or col_name not in obs.columns:
        print(f"       {label}: column not found — skipping")
        return None, None
    vals = obs[col_name].astype(str).values
    le   = LabelEncoder()
    codes = le.fit_transform(vals)
    n_cat = len(le.classes_)
    onehot = np.eye(n_cat, dtype=np.float32)[codes]
    print(f"       {label} ('{col_name}'): {n_cat} categories → "
          f"{list(le.classes_)}")
    return torch.tensor(onehot), le
 
VAX_TENSOR, VAX_LE = encode_metadata_col(adata.obs, VAX_COL, "Vaccination")
VAR_TENSOR, VAR_LE = encode_metadata_col(adata.obs, VAR_COL, "Variant")

#Scale by 0.1 to prevent large metadata values from dominating the latent space before the model has learned to use them effectively.
if VAX_TENSOR is not None:
    meta_tensors = []
if VAR_TENSOR is not None:
    meta_tensors.append(VAR_TENSOR * 0.1)
# Build combined metadata tensor [n_cells, n_meta_dims]
meta_parts = [t for t in [VAX_TENSOR, VAR_TENSOR] if t is not None]
if meta_parts:
    META_TENSOR = torch.cat(meta_parts, dim=1)
    N_META      = META_TENSOR.shape[1]
    print(f"       Total metadata dims : {N_META}")
else:
    META_TENSOR = None
    N_META      = 0
    print("       No metadata columns found — running AE-only (no Strategy A fusion)")
 
meta_encoders = {"vaccination": VAX_LE, "variant": VAR_LE}


[2/10] Encoding metadata …
       Vaccination ('Vaccination_Status'): 2 categories → ['unvaccinated', 'vaccinated']
       Variant ('Variant_Group'): 4 categories → ['Ancestral', 'Control', 'Delta', 'Omicron']
       Total metadata dims : 6


In [29]:
# %% ── Cell 5: Build GO mask ──────────────────────────────────────────────────
 
def load_gmt(library, cache_dir):
    p = Path(cache_dir) / f"{library}.pkl"
    Path(cache_dir).mkdir(exist_ok=True)
    if p.exists():
        with open(p,"rb") as f: return pickle.load(f)
    print(f"       Downloading {library} …", end=" ", flush=True)
    gs = gp.get_library(library, organism="Human")
    with open(p,"wb") as f: pickle.dump(gs, f)
    print(f"{len(gs):,} terms")
    return gs
 
print("\n[3/10] Building GO pathway mask …")
t0 = time.time()
 
all_gmt = {}
for lib in CONFIG["gmt_libraries"]:
    all_gmt.update(load_gmt(lib, CONFIG["gmt_cache_dir"]))
 
gene_names  = list(adata.var_names)
gene_set    = set(gene_names)
gene_index  = {g: i for i, g in enumerate(gene_names)}
 
X_hvg = (adata.X.toarray() if sp.issparse(adata.X)
         else adata.X).astype(np.float32)
gene_var = X_hvg.var(axis=0)
 
pathway_names, pathway_gene_lists, pathway_scores = [], [], []
for term, term_genes in all_gmt.items():
    overlap = [g for g in term_genes if g in gene_set]
    if CONFIG["min_genes_per_pathway"] <= len(overlap) <= CONFIG["max_genes_per_pathway"]:
        idx_pw = [gene_index[g] for g in overlap]
        pathway_names.append(term)
        pathway_gene_lists.append(overlap)
        pathway_scores.append(gene_var[idx_pw].mean())
 
# Keep top-N by gene variance
if CONFIG["top_n_pathways"] and CONFIG["top_n_pathways"] < len(pathway_names):
    order          = np.argsort(pathway_scores)[::-1][:CONFIG["top_n_pathways"]]
    pathway_names  = [pathway_names[i]      for i in order]
    pathway_gene_lists = [pathway_gene_lists[i] for i in order]
 
N_GENES    = len(gene_names)
N_PATHWAYS = len(pathway_names)
 
# Build boolean mask [N_PATHWAYS × N_GENES]
mask_rows = []
for pw_genes in pathway_gene_lists:
    vec = np.zeros(N_GENES, dtype=bool)
    for g in pw_genes: vec[gene_index[g]] = True
    mask_rows.append(vec)
 
MASK = torch.tensor(np.stack(mask_rows), dtype=torch.bool)
 
genes_cov = MASK.any(dim=0).sum().item()
print(f"       Pathways : {N_PATHWAYS:,}  |  Genes covered : "
      f"{genes_cov:,}/{N_GENES:,} ({genes_cov/N_GENES*100:.1f}%)")
print(f"       Mask shape: {MASK.shape}  |  Built in {time.time()-t0:.1f}s")
 
pd.Series(pathway_names, name="pathway").to_csv(
    OUTDIR / "pathways_used.csv", index=False)


[3/10] Building GO pathway mask …
       Pathways : 2,432  |  Genes covered : 2,592/5,000 (51.8%)
       Mask shape: torch.Size([2432, 5000])  |  Built in 0.4s


In [30]:
# %% ── Cell 6: Severity labels ────────────────────────────────────────────────
 
print("\n[4/10] Preparing severity labels …")
 
IS_CONTINUOUS = CONFIG["severity_type"] == "continuous"
raw_sev = adata.obs[SEVERITY_COL].values
 
if IS_CONTINUOUS:
    y_np = raw_sev.astype(np.float32)
    y_np = (y_np - y_np.mean()) / (y_np.std() + 1e-8)  # z-score for training
    y_raw_for_plot = raw_sev.astype(np.float32)          # original scale for plots
    Y_TENSOR = torch.tensor(y_np).unsqueeze(1)
    SEV_LE   = None
    N_CLASSES = 1
    print(f"       Continuous severity (z-scored): "
          f"mean={y_np.mean():.3f} std={y_np.std():.3f}")
else:
    SEV_LE  = LabelEncoder()
    y_np    = SEV_LE.fit_transform(raw_sev.astype(str)).astype(np.int64)
    Y_TENSOR = torch.tensor(y_np)
    N_CLASSES = len(SEV_LE.classes_)
    y_raw_for_plot = raw_sev
    print(f"       Categorical: {dict(enumerate(SEV_LE.classes_))}")
    print(f"       Distribution: {pd.Series(y_np).value_counts().to_dict()}")
 
X_TENSOR = torch.tensor(X_hvg)
 
# %% ── Cell 7: Patient-aware train/val split ──────────────────────────────────
 
print("\n[5/10] Train/val split (leakage-free patient split) …")
 
USE_PATIENT_SPLIT = (PATIENT_COL and PATIENT_COL in adata.obs.columns)
 
if META_TENSOR is not None:
    dataset = TensorDataset(X_TENSOR, Y_TENSOR, META_TENSOR)
else:
    dataset = TensorDataset(X_TENSOR, Y_TENSOR)
 
if USE_PATIENT_SPLIT:
    patients    = adata.obs[PATIENT_COL].values
    unique_pts  = np.unique(patients)
    rng_split   = np.random.default_rng(CONFIG["random_seed"])
    rng_split.shuffle(unique_pts)
    n_val_pts   = max(1, int(len(unique_pts) * CONFIG["val_fraction"]))
    val_pts     = set(unique_pts[:n_val_pts])
    train_idx   = np.where(~np.isin(patients, list(val_pts)))[0]
    val_idx     = np.where( np.isin(patients, list(val_pts)))[0]
    print(f"       Patient split — train: {len(train_idx):,}  "
          f"val: {len(val_idx):,}  (no leakage ✓)")
else:
    stratify = y_np if not IS_CONTINUOUS else None
    all_idx  = np.arange(len(X_TENSOR))
    train_idx, val_idx = train_test_split(
        all_idx, test_size=CONFIG["val_fraction"],
        random_state=CONFIG["random_seed"], stratify=stratify)
    print(f"       Cell split — train: {len(train_idx):,}  val: {len(val_idx):,}")
 
train_ds = Subset(dataset, train_idx)
val_ds   = Subset(dataset, val_idx)
 
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"],
                          shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"]*2,
                          shuffle=False)


[4/10] Preparing severity labels …
       Continuous severity (z-scored): mean=0.000 std=1.000

[5/10] Train/val split (leakage-free patient split) …
       Patient split — train: 8,631  val: 1,116  (no leakage ✓)


In [31]:
# %% ── Cell 8: Model (Strategy A AE + metadata concat) ───────────────────────
 
print("\n[6/10] Building model …")
 
class MaskedLinear(nn.Module):
    """GO-constrained linear layer — each neuron sees only its pathway's genes."""
    def __init__(self, in_f, out_f, mask):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_f, in_f))
        self.bias   = nn.Parameter(torch.zeros(out_f))
        self.register_buffer("mask", mask.float())
        nn.init.kaiming_uniform_(self.weight, a=0.01)
 
    def forward(self, x):
        return F.linear(x, self.weight * self.mask, self.bias)
 
 
def _block(in_d, out_d, drop, ln):
    return nn.Sequential(
        nn.Linear(in_d, out_d),
        nn.LayerNorm(out_d) if ln else nn.Identity(),
        nn.ReLU(),
        nn.Dropout(drop),
    )
 
 
class StrategyA_AE(nn.Module):
    """
    GO-masked Autoencoder with Strategy A metadata fusion.
 
    Flow
    ────
    genes → MaskedLinear → pathway scores
          → enc_hidden  → latent z [latent_dim]
 
    z || one_hot(vaccination) || one_hot(variant)   ← Strategy A concat
          → severity_head → predicted severity
 
    z → dec_hidden → pathway scores → genes         ← reconstruction
    """
    def __init__(self, n_genes, mask, pathway_hidden, latent_dim,
                 n_meta, severity_hidden, n_severity_out,
                 dropout, use_ln, continuous):
        super().__init__()
        self.continuous  = continuous
        self.n_meta      = n_meta
        n_pw             = mask.shape[0]
        z_meta_dim       = latent_dim + n_meta   # dimension after Strategy A concat
 
        # ── Encoder ──────────────────────────────────────────────────────
        self.pathway_layer = MaskedLinear(n_genes, n_pw, mask)
        self.enc_norm      = nn.Sequential(
            nn.LayerNorm(n_pw) if use_ln else nn.Identity(),
            nn.ReLU(), nn.Dropout(dropout))
        self.enc_hidden    = _block(n_pw, pathway_hidden, dropout, use_ln)
        self.enc_latent    = nn.Linear(pathway_hidden, latent_dim)
 
        # ── Strategy A: severity head (takes z + metadata) ────────────────
        # z_meta_dim = latent_dim + n_meta_onehot_dims
        # If no metadata, z_meta_dim == latent_dim (head still works)
        self.severity_head = nn.Sequential(
            _block(z_meta_dim, severity_hidden, dropout, use_ln),
            nn.Linear(severity_hidden, n_severity_out),
        )
 
        # ── Decoder ──────────────────────────────────────────────────────
        self.dec_hidden = _block(latent_dim, pathway_hidden, dropout, use_ln)
        self.dec_pw     = _block(pathway_hidden, n_pw, dropout, use_ln)
        self.dec_genes  = nn.Sequential(
            nn.Linear(n_pw, n_genes),
            nn.Softplus(),   # non-negative output matches log-normalised expression
        )
 
    def encode(self, x):
        pw = self.enc_norm(self.pathway_layer(x))
        h  = self.enc_hidden(pw)
        z  = self.enc_latent(h)
        return z, pw
 
    def decode(self, z):
        return self.dec_genes(self.dec_pw(self.dec_hidden(z)))
 
    def forward(self, x, meta=None):
        z, pw = self.encode(x)
 
        # Strategy A: concat metadata to z before severity prediction
        if meta is not None and self.n_meta > 0:
            z_meta = torch.cat([z, meta], dim=1)
        else:
            z_meta = z
 
        sev_pred = self.severity_head(z_meta)
        x_hat    = self.decode(z)
        return x_hat, z, pw, sev_pred, z_meta
 
 
MODEL = StrategyA_AE(
    n_genes         = N_GENES,
    mask            = MASK.to(DEVICE),
    pathway_hidden  = CONFIG["pathway_hidden"],
    latent_dim      = CONFIG["latent_dim"],
    n_meta          = N_META,
    severity_hidden = CONFIG["severity_hidden"],
    n_severity_out  = N_CLASSES if not IS_CONTINUOUS else 1,
    dropout         = CONFIG["dropout"],
    use_ln          = CONFIG["use_layernorm"],
    continuous      = IS_CONTINUOUS,
).to(DEVICE)
 
trainable = sum(p.numel() for p in MODEL.parameters() if p.requires_grad)
z_meta_dim = CONFIG["latent_dim"] + N_META
print(f"       Trainable params   : {trainable:,}")
print(f"       Latent dim (z)     : {CONFIG['latent_dim']}")
print(f"       Metadata dims      : {N_META}")
print(f"       z_meta dim (A)     : {z_meta_dim}  "
      f"(= {CONFIG['latent_dim']} + {N_META})")
print(f"       Mask density       : {MASK.float().mean().item()*100:.3f}%")


[6/10] Building model …
       Trainable params   : 25,605,417
       Latent dim (z)     : 32
       Metadata dims      : 6
       z_meta dim (A)     : 38  (= 32 + 6)
       Mask density       : 0.325%


In [32]:
# %% ── Cell 9: Loss functions ─────────────────────────────────────────────────
 
def recon_loss(x_hat, x):
    """
    Weighted MSE: penalises errors on high-expression genes more.
    weight = 1 + x  so a gene with expression 5 gets 6× the MSE penalty.
    This addresses the 'predicts near-zero for high-expression genes' problem
    seen in the R² scatter plot.
    """
    w = 1.0 + x
    return (w * (x_hat - x).pow(2)).mean()
 
 
def severity_loss(sev_pred, sev_true, continuous):
    """MSE for continuous WHO score; CrossEntropy for categorical."""
    if continuous:
        return F.mse_loss(sev_pred, sev_true.float())
    return F.cross_entropy(sev_pred, sev_true)

In [33]:
# %% ── Cell 10: Training loop ─────────────────────────────────────────────────
 
def run_epoch(model, loader, optimizer, train=True):
    model.train() if train else model.eval()
    tot_recon = tot_sev = tot_loss = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
 
    with ctx:
        for batch in loader:
            x   = batch[0].to(DEVICE)
            y   = batch[1].to(DEVICE)
            meta = batch[2].to(DEVICE) if len(batch) > 2 else None
 
            x_hat, z, pw, sev_pred, z_meta = model(x, meta)
 
            r_loss  = recon_loss(x_hat, x)
            s_loss  = severity_loss(sev_pred, y, model.continuous)
            loss    = CONFIG["recon_weight"] * r_loss + \
                      CONFIG["severity_weight"] * s_loss
 
            if train:
                optimizer.zero_grad()
                loss.backward()
                # Enforce GO mask — zero gradients for masked weights
                with torch.no_grad():
                    model.pathway_layer.weight.grad.mul_(
                        model.pathway_layer.mask)
                optimizer.step()
 
            tot_recon += r_loss.item()
            tot_sev   += s_loss.item()
            tot_loss  += loss.item()
 
    n = len(loader)
    return tot_recon/n, tot_sev/n, tot_loss/n
 
 
print(f"\n[7/10] Training — up to {CONFIG['epochs']} epochs …")
print(f"       recon_weight={CONFIG['recon_weight']}  "
      f"severity_weight={CONFIG['severity_weight']}")
print(f"       Early stopping: patience={CONFIG['early_stop_patience']}\n")
 
OPT  = torch.optim.Adam(MODEL.parameters(),
                         lr=CONFIG["lr"],
                         weight_decay=CONFIG["weight_decay"])
SCH  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    OPT, mode="min", factor=0.5, patience=5, verbose=False)
 
history  = {k: [] for k in ["tr_recon","vl_recon",
                              "tr_sev","vl_sev",
                              "tr_total","vl_total"]}
best_val      = float("inf")
best_path     = OUTDIR / "best_model.pt"
no_improve    = 0
stopped_at    = CONFIG["epochs"]
t0_train      = time.time()
 
for epoch in range(1, CONFIG["epochs"]+1):
    tr_r, tr_s, tr_l = run_epoch(MODEL, train_loader, OPT, train=True)
    vl_r, vl_s, vl_l = run_epoch(MODEL, val_loader,   OPT, train=False)
    SCH.step(vl_l)
 
    for k, v in zip(history.keys(), [tr_r, vl_r, tr_s, vl_s, tr_l, vl_l]):
        history[k].append(v)
 
    improved = vl_l < (best_val - CONFIG["early_stop_min_delta"])
    if improved:
        best_val = vl_l; no_improve = 0
        torch.save(MODEL.state_dict(), best_path)
    else:
        no_improve += 1
 
    if epoch % max(1, CONFIG["epochs"]//10) == 0 or epoch == 1:
        print(f"  Ep {epoch:>3}/{CONFIG['epochs']}  "
              f"recon={tr_r:.4f}/{vl_r:.4f}  "
              f"sev={tr_s:.4f}/{vl_s:.4f}  "
              f"total={tr_l:.4f}/{vl_l:.4f}  "
              f"lr={OPT.param_groups[0]['lr']:.1e}  "
              f"{'★' if improved else f'({no_improve}/{CONFIG["early_stop_patience"]})'}")
 
    if no_improve >= CONFIG["early_stop_patience"]:
        stopped_at = epoch
        print(f"\n  Early stop at epoch {epoch}")
        break
 
elapsed = time.time() - t0_train
print(f"\n  Best val loss : {best_val:.4f}")
print(f"  Stopped at    : {stopped_at}/{CONFIG['epochs']}")
print(f"  Time          : {elapsed:.1f}s  ({elapsed/stopped_at:.1f}s/epoch)")
 
MODEL.load_state_dict(torch.load(best_path, map_location=DEVICE))
MODEL.eval()


[7/10] Training — up to 50 epochs …
       recon_weight=2.0  severity_weight=1.0
       Early stopping: patience=10

  Ep   1/50  recon=0.3172/0.2831  sev=0.6437/0.6678  total=1.2782/1.2341  lr=1.0e-03  ★
  Ep   5/50  recon=0.2210/0.2178  sev=0.1628/0.6722  total=0.6047/1.1078  lr=1.0e-03  ★
  Ep  10/50  recon=0.2092/0.2054  sev=0.0983/0.7527  total=0.5167/1.1634  lr=1.0e-03  (5/10)
  Ep  15/50  recon=0.2044/0.2021  sev=0.0760/0.7116  total=0.4849/1.1159  lr=1.0e-03  (4/10)
  Ep  20/50  recon=0.1985/0.1979  sev=0.0561/0.7182  total=0.4530/1.1141  lr=5.0e-04  (9/10)

  Early stop at epoch 21

  Best val loss : 1.0405
  Stopped at    : 21/50
  Time          : 346.7s  (16.5s/epoch)


StrategyA_AE(
  (pathway_layer): MaskedLinear()
  (enc_norm): Sequential(
    (0): LayerNorm((2432,), eps=1e-05, elementwise_affine=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
  )
  (enc_hidden): Sequential(
    (0): Linear(in_features=2432, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
  )
  (enc_latent): Linear(in_features=256, out_features=32, bias=True)
  (severity_head): Sequential(
    (0): Sequential(
      (0): Linear(in_features=38, out_features=64, bias=True)
      (1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (2): ReLU()
      (3): Dropout(p=0.3, inplace=False)
    )
    (1): Linear(in_features=64, out_features=1, bias=True)
  )
  (dec_hidden): Sequential(
    (0): Linear(in_features=32, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
  )
  (de

In [34]:
# %% ── Cell 11: Extract embeddings + patient-level severity evaluation ─────────
#
# WHAT CHANGED vs original
# ────────────────────────
# Original: R² computed on ALL cells (train + val), on the model that just
#   trained on them → inflated training R², not a valid test metric.
#
# Now: Two honest evaluations:
#
#   (A) VAL-CELL R² — predictions only on held-out val cells (never seen
#       during training). Still pseudo-replicated (cells share patient labels)
#       but at least uncontaminated by training.
#
#   (B) PATIENT-LEVEL R² — for each patient, average all their cell predictions
#       into one score, then compare to true WHO. N=~17 data points.
#       This is the STATISTICALLY HONEST estimate: eliminates pseudo-replication
#       entirely. Expect wide CIs — that is correct for this sample size.
#
# Both are reported. Patient-level R² is the primary metric for your thesis.

print("\n[8/10] Extracting embeddings & patient-level evaluation ...")

# ── A: Run inference on ALL cells (for embeddings / UMAP) ────────────────────
all_z, all_pw, all_xh, all_sev, all_zmeta = [], [], [], [], []

full_ds = dataset
with torch.no_grad():
    for batch in DataLoader(full_ds, batch_size=512, shuffle=False):
        x    = batch[0].to(DEVICE)
        meta = batch[2].to(DEVICE) if len(batch) > 2 else None
        x_hat, z, pw, sev_pred, z_meta = MODEL(x, meta)
        all_xh.append(x_hat.cpu().numpy())
        all_z.append(z.cpu().numpy())
        all_pw.append(pw.cpu().numpy())
        all_sev.append(sev_pred.cpu().numpy())
        all_zmeta.append(z_meta.cpu().numpy())

Xr_np    = np.vstack(all_xh)
Z_np     = np.vstack(all_z)
PW_np    = np.vstack(all_pw)
SEV_np   = np.vstack(all_sev)
ZMETA_np = np.vstack(all_zmeta)

# Reconstruction R² (train+val — only used as AE health check, not the target metric)
recon_r2 = r2_score(X_hvg.ravel(), Xr_np.ravel())
print(f"\n  Reconstruction R² (train+val, AE health check): {recon_r2:.4f}")

# ── B: Val-cell R² (uncontaminated, still pseudo-replicated) ─────────────────
print("\n  (A) Val-cell severity R² ...")
val_sev_true = y_np[val_idx]
val_sev_pred = SEV_np[val_idx].ravel()

if IS_CONTINUOUS:
    val_cell_r2  = r2_score(val_sev_true, val_sev_pred)
    val_cell_mse = mean_squared_error(val_sev_true, val_sev_pred)

    # Back to original WHO scale for reporting
    SEV_MEAN_orig = adata.obs[SEVERITY_COL].astype(float).values.mean()
    SEV_STD_orig  = adata.obs[SEVERITY_COL].astype(float).values.std() + 1e-8
    val_who_true  = val_sev_true * SEV_STD_orig + SEV_MEAN_orig
    val_who_pred  = val_sev_pred * SEV_STD_orig + SEV_MEAN_orig
    val_who_rmse  = np.sqrt(mean_squared_error(val_who_true, val_who_pred))
    val_who_mae   = np.abs(val_who_true - val_who_pred).mean()

    print(f"     R²   = {val_cell_r2:.4f}  (z-scored)")
    print(f"     RMSE = {val_who_rmse:.3f} WHO pts")
    print(f"     MAE  = {val_who_mae:.3f} WHO pts")
    print(f"     n    = {len(val_idx):,} val cells")
    print(f"     NOTE: pseudo-replicated — cells share patient labels")
else:
    val_cell_acc = accuracy_score(val_sev_true, val_sev_pred.argmax(axis=-1)
                                  if val_sev_pred.ndim > 1 else val_sev_pred.round().astype(int))
    print(f"     Accuracy = {val_cell_acc:.4f}  ({len(val_idx):,} val cells)")

# ── C: Patient-level R² — the primary honest metric ──────────────────────────
print("\n  (B) Patient-level R² [PRIMARY METRIC] ...")
print("      Each patient's cell predictions are averaged -> one score per patient.")
print("      This eliminates pseudo-replication. Wide CI is expected and correct.\n")

if IS_CONTINUOUS and PATIENT_COL and PATIENT_COL in adata.obs.columns:
    patients_all = adata.obs[PATIENT_COL].values
    sev_raw_all  = adata.obs[SEVERITY_COL].astype(float).values

    # Restrict to val patients only (same logic as train/val split)
    val_patients = set(adata.obs[PATIENT_COL].values[val_idx])

    pt_true_who, pt_pred_who, pt_ids = [], [], []
    for pt in sorted(val_patients):
        pt_mask = (patients_all == pt)
        # Average cell predictions for this patient (back to WHO scale)
        pt_pred_z = SEV_np[pt_mask].ravel().mean()
        pt_pred_w = pt_pred_z * SEV_STD_orig + SEV_MEAN_orig
        pt_true_w = sev_raw_all[pt_mask].mean()   # should be identical per cell
        pt_true_who.append(pt_true_w)
        pt_pred_who.append(pt_pred_w)
        pt_ids.append(pt)

    pt_true = np.array(pt_true_who)
    pt_pred = np.array(pt_pred_who)

    pt_r2   = r2_score(pt_true, pt_pred)
    pt_rmse = np.sqrt(mean_squared_error(pt_true, pt_pred))
    pt_mae  = np.abs(pt_true - pt_pred).mean()
    pt_n    = len(pt_true)

    # Pearson + Spearman (appropriate for N~17)
    from scipy.stats import pearsonr, spearmanr
    pt_pearson,  pt_pearson_p  = pearsonr(pt_true, pt_pred)
    pt_spearman, pt_spearman_p = spearmanr(pt_true, pt_pred)

    # Bootstrap CI on patient-level R² (use all resamples of the ~17 patients)
    rng_boot = np.random.default_rng(42)
    boot_r2s = []
    for _ in range(2000):
        bi = rng_boot.integers(0, pt_n, pt_n)
        if len(np.unique(pt_true[bi])) < 2:
            continue
        boot_r2s.append(r2_score(pt_true[bi], pt_pred[bi]))
    boot_r2s = np.array(boot_r2s)
    ci_lo, ci_hi = np.percentile(boot_r2s, [2.5, 97.5])

    print(f"  {'Patient':<20} {'True WHO':>10} {'Pred WHO':>10} {'Error':>8}")
    print(f"  {'-'*52}")
    for pid, tw, pw_ in sorted(zip(pt_ids, pt_true, pt_pred), key=lambda x: x[1]):
        print(f"  {str(pid):<20} {tw:>10.1f} {pw_:>10.2f} {pw_-tw:>+8.2f}")
    print(f"  {'-'*52}")
    print(f"\n  Patient-level metrics  (N={pt_n} val patients):")
    print(f"    R²            = {pt_r2:.4f}")
    print(f"    95% boot CI   = [{ci_lo:.4f}, {ci_hi:.4f}]  width={ci_hi-ci_lo:.4f}")
    print(f"    RMSE          = {pt_rmse:.3f} WHO pts")
    print(f"    MAE           = {pt_mae:.3f} WHO pts")
    print(f"    Pearson  r    = {pt_pearson:.3f}  p={pt_pearson_p:.4f}")
    print(f"    Spearman rho  = {pt_spearman:.3f}  p={pt_spearman_p:.4f}")
    print(f"\n  Interpretation guide:")
    print(f"    R² > 0.3  : moderate predictive signal")
    print(f"    R² > 0.0  : better than mean-prediction baseline")
    print(f"    R² < 0.0  : model worse than predicting the mean (common at N=17)")
    print(f"    Wide CI is EXPECTED with {pt_n} patients — not a flaw")
    print(f"    Pearson/Spearman p-values more interpretable than R² at this N")

    # Save patient summary
    pt_df = pd.DataFrame({
        "patient":   pt_ids,
        "true_who":  pt_true,
        "pred_who":  pt_pred,
        "error":     pt_pred - pt_true,
        "abs_error": np.abs(pt_pred - pt_true),
    }).sort_values("true_who")
    pt_df.to_csv(OUTDIR / "patient_level_predictions.csv", index=False)
    print(f"\n  Saved patient_level_predictions.csv")

    # Primary metric for downstream use (replaces old sev_r2)
    sev_r2   = pt_r2    # patient-level is the primary metric
    sev_mse  = pt_rmse**2
    SEVERITY_METRIC = {
        "R2_patient_level": pt_r2,
        "R2_val_cell_level": val_cell_r2 if IS_CONTINUOUS else None,
        "RMSE_patient": pt_rmse,
        "MAE_patient": pt_mae,
        "Pearson_patient": pt_pearson,
        "Spearman_patient": pt_spearman,
        "N_val_patients": pt_n,
        "CI_lo": ci_lo, "CI_hi": ci_hi,
    }

    # Patient-level true vs predicted plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax = axes[0]
    ax.scatter(pt_true, pt_pred, s=80, color="#378ADD", edgecolors="white",
               linewidths=0.5, zorder=3)
    lims = [min(pt_true.min(), pt_pred.min()) - 0.5,
            max(pt_true.max(), pt_pred.max()) + 0.5]
    ax.plot(lims, lims, "r--", lw=1.2, label="perfect")
    ax.fill_between(lims, [x-1 for x in lims], [x+1 for x in lims],
                    alpha=0.08, color="green", label="+-1 WHO pt")
    for pid, tw, pw_ in zip(pt_ids, pt_true, pt_pred):
        ax.annotate(str(pid)[:8], (tw, pw_), fontsize=6,
                    xytext=(3, 3), textcoords="offset points", alpha=0.7)
    ax.set_xlabel("True WHO score (patient mean)")
    ax.set_ylabel("Predicted WHO score (cell mean)")
    ax.set_title(f"Patient-level: True vs Predicted\n"
                 f"R²={pt_r2:.3f}  Pearson r={pt_pearson:.3f}  "
                 f"p={pt_pearson_p:.3f}  N={pt_n}")
    ax.legend(fontsize=8); ax.grid(alpha=0.2)

    ax = axes[1]
    errors = pt_pred - pt_true
    colors_err = ["#E24B4A" if abs(e) > 1.5 else "#1D9E75" for e in errors]
    ax.bar(range(pt_n), errors, color=colors_err, edgecolor="white")
    ax.axhline(0,  color="black", lw=0.8, ls="--")
    ax.axhline( 1, color="green", lw=0.8, ls=":", label="+-1 WHO threshold")
    ax.axhline(-1, color="green", lw=0.8, ls=":")
    ax.set_xticks(range(pt_n))
    ax.set_xticklabels([str(p)[:8] for p in
                        [pt_ids[i] for i in range(pt_n)]],
                       rotation=45, ha="right", fontsize=7)
    ax.set_ylabel("Prediction error (pred - true WHO)")
    ax.set_title(f"Per-patient error\n"
                 f"RMSE={pt_rmse:.3f}  MAE={pt_mae:.3f} WHO pts")
    ax.legend(fontsize=8); ax.grid(alpha=0.2, axis="y")

    plt.suptitle(
        f"Patient-level severity prediction  [PRIMARY METRIC]\n"
        f"N={pt_n} val patients  |  "
        f"Immune cells only: {CONFIG['immune_filter_enabled']}  |  "
        f"95% CI=[{ci_lo:.3f},{ci_hi:.3f}]",
        fontsize=11, y=1.02)
    plt.tight_layout()
    fig.savefig(OUTDIR / "patient_level_severity.png", dpi=150, bbox_inches="tight")
    plt.show(); plt.close(fig)
    print("  Saved patient_level_severity.png")

else:
    # Categorical or no patient column: fall back to val-cell accuracy
    if not IS_CONTINUOUS:
        sev_pred_cls = SEV_np[val_idx].argmax(axis=1)
        sev_acc = accuracy_score(y_np[val_idx], sev_pred_cls)
        print(f"     Val Accuracy: {sev_acc:.4f}")
        SEVERITY_METRIC = {"Accuracy": sev_acc}
        sev_r2 = None
    else:
        print("     WARNING: PATIENT_COL not available — using val-cell R² as fallback")
        sev_r2  = val_cell_r2
        sev_mse = val_cell_mse
        SEVERITY_METRIC = {"R2_val_cells": val_cell_r2}

# ── Store in adata (unchanged — needed for UMAP etc.) ─────────────────────────
adata.obsm["X_ae_latent"]    = Z_np
adata.obsm["X_ae_pathway"]   = PW_np
adata.obsm["X_ae_zmeta"]     = ZMETA_np
adata.obs["ae_sev_predicted"] = SEV_np.ravel() if IS_CONTINUOUS \
                                 else SEV_np.argmax(axis=1).astype(str)

pd.DataFrame(PW_np, index=adata.obs_names,
             columns=pathway_names).to_csv(OUTDIR / "pathway_scores.csv")
pd.DataFrame(Z_np, index=adata.obs_names,
             columns=[f"z_{i}" for i in range(Z_np.shape[1])]).to_csv(
    OUTDIR / "latent_z.csv")
pd.DataFrame(ZMETA_np, index=adata.obs_names,
             columns=[f"zm_{i}" for i in range(ZMETA_np.shape[1])]).to_csv(
    OUTDIR / "latent_zmeta.csv")



[8/10] Extracting embeddings & predictions …
       Reconstruction R² : 0.3635
       Severity R²  : 0.8823
       Severity MSE : 0.1177


In [35]:
# %% ── Cell 12: UMAP computation ──────────────────────────────────────────────
 
print("\n[9/10] Computing UMAPs …")
 
# ── A: Gene expression (reference) ───────────────────────────────────────────
print("  A. Gene expression UMAP …")
adata_ref = adata.copy()
sc.pp.pca(adata_ref, n_comps=50)
sc.pp.neighbors(adata_ref, n_neighbors=CONFIG["umap_n_neighbors"],
                use_rep="X_pca")
sc.tl.umap(adata_ref, min_dist=CONFIG["umap_min_dist"])
UMAP_GENE = adata_ref.obsm["X_umap"]
 
# ── B: Latent z (pathway-only, no metadata) ───────────────────────────────────
print("  B. Latent z UMAP …")
sc.pp.neighbors(adata, n_neighbors=CONFIG["umap_n_neighbors"],
                use_rep="X_ae_latent")
sc.tl.umap(adata, min_dist=CONFIG["umap_min_dist"])
UMAP_Z = adata.obsm["X_umap"]
 
# ── C: z_meta (latent + vaccination + variant — Strategy A) ──────────────────
print("  C. z_meta UMAP (Strategy A — z + metadata) …")
adata_zm = adata.copy()
adata_zm.obsm["X_ae_zmeta"] = ZMETA_np
sc.pp.neighbors(adata_zm, n_neighbors=CONFIG["umap_n_neighbors"],
                use_rep="X_ae_zmeta")
sc.tl.umap(adata_zm, min_dist=CONFIG["umap_min_dist"])
UMAP_ZMETA = adata_zm.obsm["X_umap"]
 
# Store all three in adata
adata.obsm["X_umap_gene"]  = UMAP_GENE
adata.obsm["X_umap_z"]     = UMAP_Z
adata.obsm["X_umap_zmeta"] = UMAP_ZMETA


[9/10] Computing UMAPs …
  A. Gene expression UMAP …
  B. Latent z UMAP …
  C. z_meta UMAP (Strategy A — z + metadata) …


In [36]:
# %% ── Cell 13: Pearson + Spearman topology scores ────────────────────────────
 
print("\n  Computing Pearson + Spearman topology scores …")
 
def distance_correlations(emb_A, emb_B, n_sample, seed=42):
    """
    Compute Pearson r and Spearman ρ between pairwise Euclidean distances
    in two embeddings.  Self-distances (diagonal zeros) are excluded.
 
    Returns: pearson_r, pearson_p, spearman_r, spearman_p, dA, dB
    """
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(emb_A), size=min(n_sample, len(emb_A)), replace=False)
    dA  = pairwise_distances(emb_A[idx], metric="euclidean").ravel()
    dB  = pairwise_distances(emb_B[idx], metric="euclidean").ravel()
    nz  = dA > 0
    dA, dB = dA[nz], dB[nz]
    pr, pp = stats.pearsonr(dA, dB)
    sr, sp = stats.spearmanr(dA, dB)
    return pr, pp, sr, sp, dA, dB
 
 
NS = CONFIG["umap_n_sample"]
 
pr_gz, pp_gz, sr_gz, sp_gz, dG, dZ    = distance_correlations(
    UMAP_GENE, UMAP_Z,     NS)
pr_gzm, pp_gzm, sr_gzm, sp_gzm, _, dZM = distance_correlations(
    UMAP_GENE, UMAP_ZMETA, NS)
pr_self, _, sr_self, _, dGs, dGs2       = distance_correlations(
    UMAP_GENE, UMAP_GENE,  NS, seed=99)
 
# kNN label transfer
knn_acc = {}
if GROUPBY and GROUPBY in adata.obs.columns:
    cl = LabelEncoder().fit_transform(adata.obs[GROUPBY].astype(str))
    knn_acc["z"]     = KNeighborsClassifier(15).fit(
        UMAP_GENE, cl).score(UMAP_Z, cl)
    knn_acc["zmeta"] = KNeighborsClassifier(15).fit(
        UMAP_GENE, cl).score(UMAP_ZMETA, cl)
    knn_acc["self"]  = KNeighborsClassifier(15).fit(
        UMAP_GENE, cl).score(UMAP_GENE, cl)
 
print(f"\n  ── Topology metrics (n_sample={NS}) ─────────────────────────────")
print(f"  {'Comparison':<30}  {'Pearson r':>10}  {'Spearman ρ':>10}  "
      f"{'kNN acc':>8}")
print(f"  {'-'*60}")
print(f"  {'Gene ↔ Gene (ceiling)':<30}  {pr_self:>10.3f}  {sr_self:>10.3f}  "
      f"{knn_acc.get('self', float('nan')):>8.3f}")
print(f"  {'Gene ↔ Latent z':<30}  {pr_gz:>10.3f}  {sr_gz:>10.3f}  "
      f"{knn_acc.get('z', float('nan')):>8.3f}")
print(f"  {'Gene ↔ z_meta (Strategy A)':<30}  {pr_gzm:>10.3f}  {sr_gzm:>10.3f}  "
      f"{knn_acc.get('zmeta', float('nan')):>8.3f}")
print(f"  {'-'*60}")
 
improvement_pearson  = pr_gzm - pr_gz
improvement_spearman = sr_gzm - sr_gz
print(f"\n  Strategy A improvement over z-only:")
print(f"    Pearson  Δ : {improvement_pearson:+.3f}")
print(f"    Spearman Δ : {improvement_spearman:+.3f}")
 
if IS_CONTINUOUS:
    if sr_gzm > 0.5 and sev_r2 > 0.3:
        answer = "YES"
    elif sr_gzm > 0.3:
        answer = "PARTIALLY"
    else:
        answer = "NOT YET"
else:
    if sev_acc > 0.5:
        answer = "YES"
    elif sr_gzm > 0.3:
        answer = "PARTIALLY"
    else:
        answer = "NOT YET"
print(f"\n  Research question verdict: {answer}")
print(f"  → GO pathway vectors can "
      f"{'predict severity (R²={:.3f})'.format(sev_r2) if IS_CONTINUOUS else 'classify severity (acc={:.3f})'.format(sev_acc)}")
print(f"    and {'DO' if sr_gzm > 0.4 else 'DO NOT'} capture biological "
      f"complexity (Spearman ρ={sr_gzm:.3f})")
print(f"    Metadata fusion {'IMPROVES' if improvement_spearman > 0.01 else 'does not improve'} "
      f"topology (Δρ={improvement_spearman:+.3f})")


  Computing Pearson + Spearman topology scores …

  ── Topology metrics (n_sample=500) ─────────────────────────────
  Comparison                       Pearson r  Spearman ρ   kNN acc
  ------------------------------------------------------------
  Gene ↔ Gene (ceiling)                1.000       1.000     0.849
  Gene ↔ Latent z                      0.668       0.710     0.002
  Gene ↔ z_meta (Strategy A)           0.048       0.033     0.223
  ------------------------------------------------------------

  Strategy A improvement over z-only:
    Pearson  Δ : -0.621
    Spearman Δ : -0.677

  Research question verdict: NOT YET
  → GO pathway vectors can predict severity (R²=0.882)
    and DO NOT capture biological complexity (Spearman ρ=0.033)
    Metadata fusion does not improve topology (Δρ=-0.677)


In [37]:
# %% ── Cell 14: All plots ─────────────────────────────────────────────────────
 
print("\n[10/10] Plotting …")
 
sev_vals_plot = y_raw_for_plot  # original scale
 
# ── Helper ────────────────────────────────────────────────────────────────────
def scatter_umap(ax, coords, c_vals, title, cmap="RdYlBu_r",
                 categorical=False):
    if categorical:
        cats  = pd.Categorical(c_vals.astype(str))
        codes = cats.codes
        cm    = plt.cm.get_cmap("tab20", len(cats.categories))
        sc    = ax.scatter(coords[:,0], coords[:,1], c=codes, cmap=cm,
                           s=3, alpha=0.5, rasterized=True,
                           vmin=0, vmax=len(cats.categories)-1)
        if len(cats.categories) <= 12:
            hdl = [plt.Line2D([0],[0], marker='o', color='w',
                              markerfacecolor=cm(i/max(len(cats.categories)-1,1)),
                              markersize=5, label=str(c))
                   for i, c in enumerate(cats.categories)]
            ax.legend(handles=hdl, fontsize=5, loc="upper right",
                      framealpha=0.6, markerscale=1.2,
                      ncol=max(1, len(cats.categories)//6))
    else:
        sc = ax.scatter(coords[:,0], coords[:,1], c=c_vals.astype(float),
                        cmap=cmap, s=3, alpha=0.5, rasterized=True)
        plt.colorbar(sc, ax=ax, fraction=0.04, label="")
    ax.set_title(title, fontsize=9, pad=3)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel("UMAP 1", fontsize=7)
    ax.set_ylabel("UMAP 2", fontsize=7)
 
 
color_configs = [
    (SEVERITY_COL,    sev_vals_plot,              False, "WHO Score"),
    ("ae_predicted",  adata.obs["ae_sev_predicted"].values, False,  "Predicted severity"),
]
if GROUPBY    and GROUPBY    in adata.obs.columns:
    color_configs.append(
        (GROUPBY, adata.obs[GROUPBY].values, True, "Cell type"))
if VAX_COL    and VAX_COL    in adata.obs.columns:
    color_configs.append(
        (VAX_COL, adata.obs[VAX_COL].values, True, "Vaccination"))
if VAR_COL    and VAR_COL    in adata.obs.columns:
    color_configs.append(
        (VAR_COL, adata.obs[VAR_COL].values, True, "Variant"))
 
umaps = [
    ("Gene expression\n(reference)", UMAP_GENE),
    ("Latent z\n(pathway-only)",     UMAP_Z),
    ("z + metadata\n(Strategy A)",   UMAP_ZMETA),
]
 
for col_key, col_vals, is_cat, col_label in color_configs:
    if col_vals is None: continue
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (umap_title, coords) in zip(axes, umaps):
        scatter_umap(ax, coords, col_vals, umap_title, categorical=is_cat)
    plt.suptitle(f"Coloured by: {col_label}   "
                 f"({'PILOT' if CONFIG['PILOT_MODE'] else 'FULL'}, "
                 f"{adata.n_obs:,} cells)", fontsize=11)
    plt.tight_layout()
    safe = col_key.replace(" ","_").replace("/","_") if col_key else "none"
    fig.savefig(OUTDIR / f"umap_{safe}.png", dpi=150, bbox_inches="tight")
    plt.show(); plt.close(fig)
    print(f"  Saved umap_{safe}.png")
 
# ── Loss curves ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
 
ax = axes[0]
ax.plot(history["tr_recon"], label="train", lw=1.5)
ax.plot(history["vl_recon"], label="val",   lw=1.5)
ax.axvline(stopped_at-1, color="grey", lw=0.8, ls=":", label="early stop")
ax.set_title("Reconstruction loss (weighted MSE)"); ax.set_xlabel("Epoch")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
 
ax = axes[1]
ax.plot(history["tr_sev"], label="train", lw=1.5, color="coral")
ax.plot(history["vl_sev"], label="val",   lw=1.5, color="darkred")
ax.axvline(stopped_at-1, color="grey", lw=0.8, ls=":")
ax.set_title("Severity prediction loss"); ax.set_xlabel("Epoch")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
 
ax = axes[2]
gap = np.array(history["vl_total"]) - np.array(history["tr_total"])
ax.plot(gap, color="steelblue", lw=1.5)
ax.axhline(0, color="grey", lw=0.8, ls="--")
ax.axvline(stopped_at-1, color="grey", lw=0.8, ls=":")
ax.set_title("Val − Train gap\n(positive = overfitting)")
ax.set_xlabel("Epoch"); ax.grid(alpha=0.3)
 
plt.suptitle(f"Strategy A loss curves  |  "
             f"Recon R²={recon_r2:.3f}  |  "
             f"{'Sev R²' if IS_CONTINUOUS else 'Sev Acc'}="
             f"{list(SEVERITY_METRIC.values())[0]:.3f}",
             fontsize=11, y=1.01)
plt.tight_layout()
fig.savefig(OUTDIR / "loss_curves.png", dpi=150, bbox_inches="tight")
plt.show(); plt.close(fig)
print("  Saved loss_curves.png")
 
# ── Severity scatter (true vs predicted) ─────────────────────────────────────
if IS_CONTINUOUS:
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(sev_true_full, SEV_np.ravel(), s=3, alpha=0.4, color="#378ADD")
    lims = [min(sev_true_full.min(), SEV_np.min()),
            max(sev_true_full.max(), SEV_np.max())]
    ax.plot(lims, lims, "r--", lw=0.8)
    ax.set_xlabel("True severity (z-scored)")
    ax.set_ylabel("Predicted severity")
    ax.set_title(f"Severity prediction  R²={sev_r2:.3f}")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(OUTDIR / "severity_scatter.png", dpi=150, bbox_inches="tight")
    plt.show(); plt.close(fig)
    print("  Saved severity_scatter.png")
 
# ── Topology correlation scatter ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (dA, dB, title, pr, sr) in zip(axes, [
    (dG,  dZ,  "Gene ↔ Latent z",           pr_gz,  sr_gz),
    (dG,  dZM, "Gene ↔ z_meta (Strategy A)", pr_gzm, sr_gzm),
]):
    rng_sc = np.random.default_rng(0)
    si     = rng_sc.choice(len(dA), min(3000, len(dA)), replace=False)
    ax.scatter(dA[si], dB[si], s=1.5, alpha=0.2, color="#378ADD", rasterized=True)
    m, b = np.polyfit(dA, dB, 1)
    xl   = np.array([dA.min(), dA.max()])
    ax.plot(xl, m*xl+b, "r-",  lw=1.5, label=f"Pearson r={pr:.3f}")
    ax.plot(xl, xl*(dB.mean()/max(dA.mean(),1e-8)), "g--",
            lw=1.0, alpha=0.7, label=f"Spearman ρ={sr:.3f}")
    ax.set_xlabel("Pairwise dist — Gene UMAP", fontsize=10)
    ax.set_ylabel("Pairwise dist — other UMAP", fontsize=10)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9); ax.grid(alpha=0.2)
plt.suptitle("UMAP topology preservation\n"
             f"(n={NS} cells, self ceiling: "
             f"Pearson={pr_self:.3f}, Spearman={sr_self:.3f})",
             fontsize=11)
plt.tight_layout()
fig.savefig(OUTDIR / "topology_correlation.png", dpi=150, bbox_inches="tight")
plt.show(); plt.close(fig)
print("  Saved topology_correlation.png")
 
# ── SHAP (if available) ───────────────────────────────────────────────────────
if HAS_SHAP and N_PATHWAYS > 0:
    print("\n  SHAP pathway importance …")
    try:
        # Explain severity head: input = z_meta, output = severity prediction
        class SevHead(nn.Module):
            def __init__(self, model): super().__init__(); self.m = model
            def forward(self, zm): return self.m.severity_head(zm)
 
        sev_module = SevHead(MODEL).to(DEVICE).eval()
        n_bg  = min(100, len(ZMETA_np))
        n_exp = min(CONFIG["shap_n_cells"], len(ZMETA_np))
        rng_s = np.random.default_rng(CONFIG["random_seed"])
 
        bg   = torch.tensor(ZMETA_np[rng_s.choice(len(ZMETA_np), n_bg,
                             replace=False)],
                             dtype=torch.float32).to(DEVICE)
        exp  = torch.tensor(ZMETA_np[rng_s.choice(len(ZMETA_np), n_exp,
                             replace=False)],
                             dtype=torch.float32).to(DEVICE)
 
        explainer   = shap.DeepExplainer(sev_module, bg)
        shap_values = explainer.shap_values(exp)
 
        if isinstance(shap_values, list):
            sv = np.abs(np.stack(shap_values)).mean(axis=0)
        elif hasattr(shap_values, "ndim") and shap_values.ndim == 3:
            sv = np.abs(shap_values).mean(axis=2)
        else:
            sv = np.abs(shap_values)
 
        mean_shap = sv.mean(axis=0)   # [latent_dim + n_meta]
 
        # Build feature names: latent dims + metadata category names
        meta_names = []
        if VAX_LE: meta_names += [f"Vax:{c}" for c in VAX_LE.classes_]
        if VAR_LE: meta_names += [f"Var:{c}" for c in VAR_LE.classes_]
        feat_names = [f"z_{i}" for i in range(CONFIG["latent_dim"])] + meta_names
 
        top_n   = min(25, len(feat_names))
        top_idx = [int(i) for i in np.argsort(mean_shap)[::-1][:top_n]]
        fs      = pd.Series(feat_names)
        top_lbl = [fs.iloc[i] for i in top_idx]
        top_sc  = mean_shap[top_idx]
 
        fig, ax = plt.subplots(figsize=(10, 7))
        colors  = plt.cm.RdYlBu_r(np.linspace(0.1, 0.9, top_n))
        ax.barh(top_lbl, top_sc, color=colors)
        ax.invert_yaxis()
        ax.set_xlabel("Mean |SHAP value|")
        ax.set_title("SHAP importance: latent dims + metadata → severity")
        plt.tight_layout()
        fig.savefig(OUTDIR / "shap_severity.png", dpi=150, bbox_inches="tight")
        plt.show(); plt.close(fig)
        print("  Saved shap_severity.png")
 
        pd.DataFrame({
            "feature": feat_names,
            "mean_abs_shap": mean_shap,
        }).sort_values("mean_abs_shap", ascending=False).to_csv(
            OUTDIR / "shap_scores.csv", index=False)
    except Exception as e:
        print(f"  SHAP failed: {e}")


[10/10] Plotting …
  Saved umap_WHO_Score_at_Peak.png
  Saved umap_ae_predicted.png
  Saved umap_Coarse_Annotation.png
  Saved umap_Vaccination_Status.png
  Saved umap_Variant_Group.png
  Saved loss_curves.png
  Saved severity_scatter.png
  Saved topology_correlation.png

  SHAP pathway importance …
  SHAP failed: The SHAP explanations do not sum up to the model's output! This is either because of a rounding error or because an operator in your computation graph was not fully supported. If the sum difference of %f is significant compared to the scale of your model outputs, please post as a github issue, with a reproducible example so we can debug it. Used framework: pytorch - Max. diff: 0.6739298271859298 - Tolerance: 0.01


In [38]:

# %% ── Cell 15: Save metrics + research question summary ─────────────────────
 
adata.write_h5ad(OUTDIR / "adata_strategy_a.h5ad")
torch.save({
    "state_dict":    MODEL.state_dict(),
    "pathway_names": pathway_names,
    "gene_names":    gene_names,
    "config":        CONFIG,
    "mask":          MASK,
    "vax_le":        VAX_LE,
    "var_le":        VAR_LE,
    "sev_le":        SEV_LE,
}, OUTDIR / "strategy_a_checkpoint.pt")
 
all_metrics = {
    # Research question core metrics
    "research_q_severity_r2":          sev_r2 if IS_CONTINUOUS else None,
    "research_q_severity_acc":         None if IS_CONTINUOUS else sev_acc,
    "research_q_recon_r2":             recon_r2,
    # Topology
    "topology_pearson_z":              pr_gz,
    "topology_spearman_z":             sr_gz,
    "topology_pearson_zmeta":          pr_gzm,
    "topology_spearman_zmeta":         sr_gzm,
    "topology_pearson_ceiling":        pr_self,
    "topology_spearman_ceiling":       sr_self,
    "topology_pearson_pct_ceiling":    pr_gzm/pr_self*100 if pr_self else None,
    "topology_spearman_pct_ceiling":   sr_gzm/sr_self*100 if sr_self else None,
    # Strategy A improvement
    "strategy_a_pearson_delta":        improvement_pearson,
    "strategy_a_spearman_delta":       improvement_spearman,
    # kNN
    "knn_acc_z":                       knn_acc.get("z"),
    "knn_acc_zmeta":                   knn_acc.get("zmeta"),
    "knn_acc_ceiling":                 knn_acc.get("self"),
    # Model
    "n_pathways":                      N_PATHWAYS,
    "n_genes_hvg":                     N_GENES,
    "n_meta_dims":                     N_META,
    "n_cells_pilot":                   adata.n_obs,
    "stopped_epoch":                   stopped_at,
    "trainable_params":                trainable,
    "vaccination_col":                 str(VAX_COL),
    "variant_col":                     str(VAR_COL),
    "severity_col":                    SEVERITY_COL,
}
pd.Series(all_metrics).to_csv(OUTDIR / "research_metrics.csv", header=["value"])
 
print(f"\n{'='*62}")
print(f"  STRATEGY A — RESEARCH QUESTION SUMMARY")
print(f"{'='*62}")
print(f"  Question: Can GO pathway vectors + AE predict COVID severity?")
print(f"")
print(f"  Severity prediction")
print(f"    Metric : {'R²' if IS_CONTINUOUS else 'Accuracy'} = "
      f"{list(SEVERITY_METRIC.values())[0]:.3f}")
print(f"    Verdict: {'SUPPORTS' if list(SEVERITY_METRIC.values())[0] > 0.3 else 'WEAK'} "
      f"the research hypothesis")
print(f"")
print(f"  Biological complexity (UMAP topology)")
print(f"    Latent z     : Spearman ρ = {sr_gz:.3f}  "
      f"Pearson r = {pr_gz:.3f}")
print(f"    z + metadata : Spearman ρ = {sr_gzm:.3f}  "
      f"Pearson r = {pr_gzm:.3f}")
print(f"    Improvement  : Δρ = {improvement_spearman:+.3f}")
print(f"    Verdict: metadata fusion "
      f"{'IMPROVES' if improvement_spearman > 0.01 else 'does not improve'} "
      f"topology capture")
print(f"")
print(f"  Outputs → {OUTDIR.resolve()}")
print(f"    loss_curves.png            — recon + severity + gap")
print(f"    umap_*.png                 — UMAPs for all colour keys")
print(f"    topology_correlation.png   — Pearson + Spearman distance scatter")
print(f"    severity_scatter.png       — true vs predicted severity")
print(f"    shap_severity.png/.csv     — feature importance")
print(f"    research_metrics.csv       — all quantitative results")
print(f"    adata_strategy_a.h5ad      — AnnData + all embeddings")
print(f"{'='*62}")


  STRATEGY A — RESEARCH QUESTION SUMMARY
  Question: Can GO pathway vectors + AE predict COVID severity?

  Severity prediction
    Metric : R² = 0.882
    Verdict: SUPPORTS the research hypothesis

  Biological complexity (UMAP topology)
    Latent z     : Spearman ρ = 0.710  Pearson r = 0.668
    z + metadata : Spearman ρ = 0.033  Pearson r = 0.048
    Improvement  : Δρ = -0.677
    Verdict: metadata fusion does not improve topology capture

  Outputs → C:\Users\ulago\Downloads\Applied-Machine-Learning-Final-Project\strategy_a_results\pilot
    loss_curves.png            — recon + severity + gap
    umap_*.png                 — UMAPs for all colour keys
    topology_correlation.png   — Pearson + Spearman distance scatter
    severity_scatter.png       — true vs predicted severity
    shap_severity.png/.csv     — feature importance
    research_metrics.csv       — all quantitative results
    adata_strategy_a.h5ad      — AnnData + all embeddings
